# 01 — Ingestão, validação e pré-processamento do dataset Beans

**Data Challenge | Desafio 1: Classificação de doenças em folhas**

Este notebook cobre a primeira etapa do projeto: carregar o dataset do Hugging Face, inspecionar os splits, validar as classes e construir um pipeline inicial de redimensionamento e normalização das imagens.


## 1. Problema e objetivo

O problema é uma **classificação multiclasse de imagens**. A entrada é uma fotografia de folha de feijão e a saída esperada é uma entre três classes: `angular_leaf_spot`, `bean_rust` ou `healthy`.

Nesta primeira etapa, o objetivo técnico é garantir uma ingestão reprodutível dos dados e preparar um pipeline de imagens consistente para os experimentos de EDA e modelagem que virão na sequência.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from bean_leaf_disease.data.beans import (
    class_distribution,
    class_names,
    load_beans_dataset,
    make_tf_dataset,
    split_summary,
)
from bean_leaf_disease.utils.reproducibility import set_global_seed

SEED = 42
set_global_seed(SEED)
pd.set_option('display.max_columns', None)


## 2. Ingestão do dataset via Hugging Face

In [ ]:
dataset = load_beans_dataset()
dataset

In [ ]:
summary = split_summary(dataset)
summary

## 3. Classes e distribuição por split

Antes de modelar, verificamos se os rótulos disponíveis correspondem às três classes esperadas e se existe algum indício de desbalanceamento entre treino, validação e teste.

In [ ]:
labels = class_names(dataset)
labels

In [ ]:
distribution = class_distribution(dataset)
distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=distribution, x='split', y='count', hue='class_name')
plt.title('Distribuição de classes por split')
plt.xlabel('Split')
plt.ylabel('Quantidade de imagens')
plt.tight_layout()
plt.show()

## 4. Inspeção visual de amostras

A inspeção visual ajuda a verificar resolução, enquadramento, iluminação e variabilidade antes de definir augmentation e arquitetura de rede.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, class_id in enumerate(range(len(labels))):
    class_samples = [s for s in dataset['train'] if s['labels'] == class_id][:4]
    for col, sample in enumerate(class_samples):
        axes[row, col].imshow(sample['image'])
        axes[row, col].set_title(labels[class_id])
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 5. Pipeline inicial: resize + normalização

Para padronizar a entrada dos modelos, adotamos inicialmente `224 × 224` pixels e normalização dos pixels para o intervalo `[0, 1]`. O valor 224 é compatível com várias arquiteturas populares de Transfer Learning e permite reaproveitar este pipeline depois.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = make_tf_dataset(
    dataset['train'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True, seed=SEED
)
val_ds = make_tf_dataset(
    dataset['validation'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
test_ds = make_tf_dataset(
    dataset['test'], image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)

In [ ]:
images, y = next(iter(train_ds))
print('batch images:', images.shape)
print('batch labels:', y.shape)
print('min pixel:', float(images.numpy().min()))
print('max pixel:', float(images.numpy().max()))

## 6. Checks de qualidade iniciais

Estes checks servem para detectar problemas antes de qualquer treinamento: splits vazios, classes inesperadas e falhas no pipeline de imagem.

In [ ]:
assert set(dataset.keys()) >= {'train', 'validation', 'test'}
assert len(labels) == 3
assert images.shape[1:] == (224, 224, 3)
assert 0.0 <= float(images.numpy().min())
assert float(images.numpy().max()) <= 1.0
print('Checks iniciais concluídos com sucesso.')

## 7. Artefatos para os próximas etapas

O notebook deixa prontos: (1) carregamento do dataset, (2) nomes das classes, (3) distribuição por split e (4) datasets TensorFlow pré-processados. Isso permite que a EDA continue sobre a base original e que o baseline consuma diretamente `train_ds`, `val_ds` e `test_ds`.


In [ ]:
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
distribution.to_csv(output_dir / 'class_distribution.csv', index=False)
summary.to_csv(output_dir / 'split_summary.csv', index=False)
print('Metadados salvos em data/processed/.')